In [ ]:
%matplotlib widget
import os
from ipyfilechooser import FileChooser
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy
import pandas as pd

import scipp as sc
import plopp

import magic_graphs
import read_h5
import operations_with_da
import peak_find
import get_ub


# Single Crystal Data Reduction Workflow for MAGiC diffractometer

This notebook presents skeleton of data reduction procedure for Single Crystal for McStas h5 file of MAGiC using SCIPP. The steps are described on the corresponding [confluence page](https://confluence.ess.eu/spaces/~iuriikibalin/pages/788704198/Single-Crystal+Data+Reduction+Workflow+for+MAGiC+with+Scipp+Tasks).

![scheme_magic](scheme.png "Scheme of MAGiC diffractometer")

In [ ]:
def unique_random_integers(N, m):
    max_size = N + 1
    size = min(m, max_size)
    return numpy.random.choice(max_size, size=size, replace=False)

def randomly_take_n_events(da, n):
    flag_border = sc.zeros(dims=da.dims, shape= da.shape, dtype=bool)
    for key in da.masks.keys():
        flag_border |= da.masks['key']
    da_not_border = da[~flag_border]    
    np_arg = unique_random_integers(da_not_border.size-1, n)
    return da_not_border[np_arg]

In [ ]:
# ---------------------------------------------------------
# Dummy peak finder (replace with your real function)
# ---------------------------------------------------------
def find_peaks(data_event):
    """
    Replace this with your real peak-finding algorithm.
    Must return something displayable (DataFrame, dict, scipp table, etc.)
    """
    data_event_random = randomly_take_n_events(data_event, 100000)
    
    data_event_random = data_event_random.transform_coords(("Q_vec_rot",), graph=magic_graphs.graph_qvec)
    
    da_peaks = peak_find.find_multiple_peaks_accel(
    events_coords=data_event_random.coords['Q_vec_rot'],
    events_weight=data_event_random.data,
    merge_radius=0.1,
    basin_radius=0.2,
    max_seeds=5000,
    random_state=None,
    radius_factor=3.0,
    )

    return da_peaks


In [ ]:
def display_center(*graphs):
    return display(widgets.VBox(graphs, layout=widgets.Layout(align_items='center')))

In [ ]:
DATA_STORAGE = {}


def calc_normalization_per_monitor():
    data_event = DATA_STORAGE['data_event'] # b.data_event
    data_cave_monitor = DATA_STORAGE['data_cave_monitor']
    data_event_normalized = operations_with_da.normalize_per_cave_monitor(data_event, data_cave_monitor)
    DATA_STORAGE['data_event_normalized_per_monitor'] = data_event_normalized

def calc_normalization_per_vanadium():
    print("Normalization per vanadium is not currently implemented")
    data_event = DATA_STORAGE['data_event'] 
    data_event_vanadium = DATA_STORAGE['data_event_vanadium'] 
    # data_event_normalized = operations_with_da.normalize_per_vanadium(data_event, data_event_vanadium)
    DATA_STORAGE['data_event_normalized_per_vanadium'] = data_event


# ---------------------------------------------------------
# File chooser widget
# ---------------------------------------------------------
fc = FileChooser(
    '/Users/iuriikibalin/Repositories/magic/run_folder',
    title='Select NEXuS file with Single crystal diffraction data',
    select_default=False,
)

fc_vanadium = FileChooser(
    '/Users/iuriikibalin/Repositories/magic/run_folder',
    title='Select NEXuS file with Vanadium measurements for normatlization (optional)',
    select_default=False,
)




load_events_button = widgets.Button(
    description="Load",
    button_style='success'
)

display_button = widgets.Button(
    description="Display",
    button_style='info',
    layout=widgets.Layout(display='none')  # hidden initially
)


# New button (initially hidden)
find_peaks_button = widgets.Button(
    description="Find Peaks",
    button_style='success',
    layout=widgets.Layout(display='none')  # hidden initially
)

fig_rbuttons = widgets.RadioButtons(
    options=['3D Laue Pattern', '2D Pattern', 'Monitor Data', '3D Normalization Data', '2D Normalization Data'],
    value='3D Laue Pattern', # Defaults to 'pineapple'
#    layout={'width': 'max-content'}, # If the items' names are long
    description='Data to display:',
    disabled=False,
    layout=widgets.Layout(display='none')  # hidden initially
)


norm_rbuttons = widgets.ToggleButtons(
    options=['No Normalization', 'Per Monitor', 'Per Vanadium'],
    description='Choose Normalization Condition',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltips=['No normalization', 'Per Monitor', 'Per Vanadium'],
#     icons=['check'] * 3
)

cell_a = widgets.BoundedFloatText(
    value=7.5,
    min=0,
    max=60.0,
    step=0.0001,
    description='a:',
    disabled=False,
    layout=widgets.Layout(display='none')  # hidden initially
)
cell_b = widgets.BoundedFloatText(
    value=7.5,
    min=0,
    max=60.0,
    step=0.0001,
    description='b:',
    disabled=False,
    layout=widgets.Layout(display='none')  # hidden initially
)
cell_c = widgets.BoundedFloatText(
    value=7.5,
    min=0,
    max=60.0,
    step=0.0001,
    description='c:',
    disabled=False,
    layout=widgets.Layout(display='none')  # hidden initially
)
cell_alpha = widgets.BoundedFloatText(
    value=90,
    min=0.1,
    max=179.9,
    step=0.01,
    description='Alpha:',
    disabled=False,
    layout=widgets.Layout(display='none')  # hidden initially
)
cell_beta = widgets.BoundedFloatText(
    value=90,
    min=0.1,
    max=179.9,
    step=0.0001,
    description='Beta:',
    disabled=False,
    layout=widgets.Layout(display='none')  # hidden initially
)
cell_gamma = widgets.BoundedFloatText(
    value=90,
    min=0.1,
    max=179.9,
    step=0.01,
    description='Gamma:',
    disabled=False,
    layout=widgets.Layout(display='none')  # hidden initially
)

indexation_button = widgets.Button(
    description="Indexate peaks (find UB)",
    button_style='info',
    layout=widgets.Layout(display='none')  # hidden initially
)


# output = widgets.Output()
output_data = widgets.Output()
peaks_output = widgets.Output()  # separate area for peaks table
indexation_output = widgets.Output()  # separate area for indexation


# ---------------------------------------------------------
# Callback: load + plot
# ---------------------------------------------------------
def load_file(b):
    with output_data:
        clear_output()
        DATA_STORAGE.clear()
        peaks_output.clear_output()
        fig_rbuttons.layout.display = 'none'
        display_button.layout.display = 'none'
        norm_rbuttons.layout.display = 'none'
        find_peaks_button.layout.display = 'none'  # hide until plot is ready
        cell_a.layout.display = 'none'
        cell_b.layout.display = 'none'
        cell_c.layout.display = 'none'
        cell_alpha.layout.display = 'none'
        cell_beta.layout.display = 'none'
        cell_gamma.layout.display = 'none'
        indexation_button.layout.display = 'none'

        file_path = fc.selected
        if file_path is None or not os.path.isfile(file_path):
            print("No file selected")
            return
        try:
            d_out = read_h5.read_h5_to_dict(file_path)
        except Exception as e:
            print(f"Error reading file: {e}")
            return

        DATA_STORAGE['data_event'] = d_out['data_event']
        DATA_STORAGE['data_cave_monitor'] = d_out['data_cave_monitor']


        file_path = fc_vanadium.selected
        if not file_path is None and os.path.isfile(file_path):
            try:
                d_out = read_h5.read_h5_to_dict(file_path)
                DATA_STORAGE['data_event_vanadium'] = d_out['data_event']
                DATA_STORAGE['data_cave_monitor_vanadium'] = d_out['data_cave_monitor']
            except Exception as e:
                print(f"Error reading file: {e}")
            
        plot_file(b)
        fig_rbuttons.layout.display = 'inline-block'
        display_button.layout.display = 'inline-block'
        norm_rbuttons.layout.display = 'inline-block'
        find_peaks_button.layout.display = 'inline-block'


def plot_file(b):
    with output_data:
        clear_output()
        # peaks_output.clear_output()
        # find_peaks_button.layout.display = 'none'  # hide until plot is ready


        # Create plopp figure
        print(fig_rbuttons.value)
        flag_display_center = True
        if fig_rbuttons.value == '3D Laue Pattern':
            data_event = DATA_STORAGE['data_event']
            da_laue = operations_with_da.da_to_laue_hist(data_event, factor_border=0.07)
            vmax = numpy.quantile(da_laue.data.values,0.9)
            fig = plopp.scatter3d(da_laue, pos='detector_event_position_local', cbar=True, size=0.005, opacity=0.75, vmax=vmax)
        elif fig_rbuttons.value == '2D Pattern':
            data_event = DATA_STORAGE['data_event']
            da_2d = operations_with_da.da_to_2d_hist(data_event, factor_border=0.07)
            fig = plopp.inspector(da_2d, dim='toa', orientation='vertical', logc=False, mode='rectangle')
        elif fig_rbuttons.value == '3D Normalization Data':
            data_event = DATA_STORAGE['data_event_vanadium']
            da_laue = operations_with_da.da_to_laue_hist(data_event, factor_border=0.07)
            vmax = numpy.quantile(da_laue.data.values,0.9)
            fig = plopp.scatter3d(da_laue, pos='detector_event_position_local', cbar=True, size=0.005, opacity=0.75, vmax=vmax)
        elif fig_rbuttons.value == '2D Normalization Data':
            data_event = DATA_STORAGE['data_event_vanadium']
            da_2d = operations_with_da.da_to_2d_hist(data_event, factor_border=0.07)
            fig = plopp.inspector(da_2d, dim='toa', orientation='vertical', logc=False, mode='rectangle')
        elif fig_rbuttons.value == 'Monitor Data':
            data_cave_monitor = DATA_STORAGE['data_cave_monitor']
            fig = data_cave_monitor.hist(toa=101).plot()
        else:
            flag_display_center = False
            fig, ax = plt.subplots()
            ax.plot([1, 2, 3], [4, 5, 6])
            
        if flag_display_center:
            display_center(fig)
        else:
            display(fig)

        # Store data_event for peak finding
        # find_peaks_button.data_event = data_event

        # Show the Find Peaks button

# ---------------------------------------------------------
# Callback: find peaks
# ---------------------------------------------------------
def run_peak_finder(b):
    with peaks_output:
        clear_output()

        if norm_rbuttons.value == "No Normalization":
            data_event = DATA_STORAGE['data_event'] # b.data_event
        elif norm_rbuttons.value == "Per Monitor":
            if not 'data_event_normalized_per_monitor' in DATA_STORAGE.keys():
                calc_normalization_per_monitor()
            data_event = DATA_STORAGE['data_event_normalized_per_monitor'] # b.data_event
        elif norm_rbuttons.value == "Per Vanadium":
            if not 'data_event_normalized_per_vanadium' in DATA_STORAGE.keys():
                calc_normalization_per_vanadium()
            data_event = DATA_STORAGE['data_event_normalized_per_vanadium'] # b.data_event
        else:
            print("Something is wrong in run_peak_finder")
            return
        
        peaks = find_peaks(data_event)

        print("Detected peaks:")
        DATA_STORAGE['data_peaks'] = peaks

        np_q = peaks.coords['Q_vec_rot'].values
        pd_peaks = pd.DataFrame({
        "Qx": np_q[:,0],
        "Qy": np_q[:,1],
        "Qz": np_q[:,2],
        "Intensity": peaks.data.values
        })
        fig = plopp.scatter3d(peaks, pos='Q_vec_rot', cbar=True, size=1, perspective=False)
    
        
        display(pd_peaks, fig)
        cell_a.layout.display = 'inline-block' 
        cell_b.layout.display = 'inline-block' 
        cell_c.layout.display = 'inline-block' 
        cell_alpha.layout.display = 'inline-block' 
        cell_beta.layout.display = 'inline-block' 
        cell_gamma.layout.display = 'inline-block' 
        indexation_button.layout.display = 'inline-block' 

def indexate_peaks(b):
    with indexation_output:
        da_peaks = DATA_STORAGE['data_peaks']
        
        clear_output()
        cell_a = sc.scalar(14.04078, unit="angstrom")
        cell_b = sc.scalar(14.04078, unit="angstrom")
        cell_c = sc.scalar(14.04078, unit="angstrom")
        cell_alpha = sc.scalar(90., unit="deg")
        cell_beta = sc.scalar(90., unit="deg")
        cell_gamma = sc.scalar(90., unit="deg")

        # First estimation
        euler_alpha = sc.scalar(1., unit="deg")
        euler_beta = sc.scalar(1., unit="deg")
        euler_gamma = sc.scalar(0., unit="deg")

        # Only strong peaks used for refinement
        factor = 0.3
        da_peaks_strong = da_peaks[da_peaks.data > factor* da_peaks.data.max()]
        print("Number of stronger peaks", da_peaks_strong.size)

        print("# No refinement UB-matrix")
        ea_opt, sc_b_matrix, chi_sq = get_ub.get_euleur_opt(
            cell_a, cell_b, cell_c, cell_alpha, cell_beta, cell_gamma, 
            da_peaks_strong.coords["Q_vec_rot"], da_peaks_strong.data.values,
            euler_alpha, euler_beta, euler_gamma, graph_hkl=magic_graphs.graph_hkl,
            relfine_unit_cell=False, singony='cubic')
        euler_alpha, euler_beta, euler_gamma = ea_opt[0],ea_opt[1],ea_opt[2]
        print(f"Optimized Euler angles (deg):\n {ea_opt[0].to(unit='deg').value:.2f} {ea_opt[1].to(unit='deg').value:.2f} {ea_opt[2].to(unit='deg').value:.2f}\n")
        print(f"Chi-squared: {chi_sq:.4f}\n")
    
        print("# UB-matrix is refined")
        ea_opt, sc_b_matrix, chi_sq = get_ub.get_euleur_opt(
            cell_a, cell_b, cell_c, cell_alpha, cell_beta, cell_gamma, 
            da_peaks_strong.coords["Q_vec_rot"], da_peaks_strong.data.values,
            euler_alpha, euler_beta, euler_gamma, graph_hkl=magic_graphs.graph_hkl,
            relfine_unit_cell=True, singony='cubic')

        print(f"Optimized Euler angles (deg):\n {ea_opt[0].to(unit='deg').value:.2f} {ea_opt[1].to(unit='deg').value:.2f} {ea_opt[2].to(unit='deg').value:.2f}\n")
        print(f"Optimized B matrix:\n{sc_b_matrix.values}\n")
        unit_cell = magic_graphs.graph_ub_inv[("cell_a", "cell_b", "cell_c", "cell_alpha", "cell_beta", "cell_gamma")](sc_b_matrix)
        ls_out=["Optimized unit cell:",]
        l_label = ["a","b","c","alpha","beta","gamma"]
        for ind, label in enumerate(l_label):
            ls_out.append(f"{label:>10} : {unit_cell[ind].value:} {unit_cell[ind].unit:}")
        print("\n".join(ls_out)+"\n")

        print(f"Chi-squared: {chi_sq:.4f}\n")


            
find_peaks_button.on_click(run_peak_finder)
load_events_button.on_click(load_file)
display_button.on_click(plot_file)
indexation_button.on_click(indexate_peaks)

fig_rbuttons.layout.display = 'none'
display_button.layout.display = 'none'
find_peaks_button.layout.display = 'none'   
norm_rbuttons.layout.display = 'none'

# ---------------------------------------------------------
# Display UI
# ---------------------------------------------------------
ui_fc = widgets.HBox(
    [fc, fc_vanadium],
    layout=widgets.Layout(align_items='center')
)


ui_control_display = widgets.HBox(
    [fig_rbuttons, display_button],
    layout=widgets.Layout(align_items='center')
)

ui_abc = widgets.HBox(
    [cell_a,cell_b,cell_c,],
    layout=widgets.Layout(align_items='center')
)

ui_angles = widgets.HBox(
    [cell_alpha,cell_beta,cell_gamma,],
    layout=widgets.Layout(align_items='center')
)

display_center(ui_fc, load_events_button, ui_control_display, output_data, norm_rbuttons, find_peaks_button, peaks_output, ui_abc, ui_angles, indexation_button, indexation_output)
